# 08-2 Choropleth 地圖：台灣退伍軍人病縣市分布（2003–至今）

護理之家內部分析用 heatmap 和 spot map；若要到社區層級，就需要 **Choropleth（分級著色地圖）**。

本 notebook 使用兩個真實政府開放資料：
1. **國土測繪中心（NLSC）** — 直轄市、縣市界線 SHP（TWD97 EPSG:3824）
2. **疾管署（CDC）** — 2003 年起台灣退伍軍人病地區年齡性別統計表

流程：**下載資料 → 讀取 SHP → 讀取 CDC CSV → 台/臺 正規化 → ID 比對 → 靜態地圖 → 年度動畫**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys, os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e . pillow

In [ ]:
# --- 套件與字型設定 ---
import pathlib, urllib.request, zipfile, warnings, ssl
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.animation import FuncAnimation
from IPython.display import Image as IPyImage

for _fd in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _fd.exists():
        for _fp in sorted(_fd.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name):
                try: fm.fontManager.addfont(str(_fp))
                except Exception: pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS", "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore")
print("套件載入完成")

In [ ]:
# --- Step 1: 下載政府開放資料 ---
DATA_DIR    = pathlib.Path("data/external")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 國土測繪中心 縣市界線 (TWD97 EPSG:3824)
SHP_ZIP_URL = (
    "https://maps.nlsc.gov.tw/download/"
    "%E7%B8%A3%E5%B8%82%E7%95%8C%E7%B7%9A"
    "(TWD97%E7%B6%93%E7%B7%AF%E5%BA%A6).zip"
)
# 疾管署 退伍軍人病 地區年齡性別統計（發病日月統計，2003–）
CDC_CSV_URL = "https://od.cdc.gov.tw/eic/Age_County_Gender_4828.csv"

SHP_ZIP = DATA_DIR / "tw_county_boundary.zip"
SHP_DIR = DATA_DIR / "tw_county_shp"
CDC_CSV = DATA_DIR / "tw_legionella_4828.csv"


def _ssl_ctx(insecure: bool = False) -> ssl.SSLContext:
    """建立 SSL context；insecure=True 時停用憑證驗證（給憑證格式不合規的政府站用）。"""
    if insecure:
        ctx = ssl.create_default_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        return ctx
    return ssl.create_default_context()


def _download(url: str, dest: pathlib.Path, timeout: int = 45) -> bool:
    """下載 url → dest；已存在則跳過；SSL 驗證失敗時自動降級重試。"""
    if dest.exists():
        print(f"✓ 已有快取：{dest.name}")
        return True
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    # 第一次：嚴格驗證；第二次：停用驗證（針對台灣政府站憑證缺 SKID 問題）
    for insecure in (False, True):
        try:
            with urllib.request.urlopen(req, timeout=timeout, context=_ssl_ctx(insecure)) as resp:
                dest.write_bytes(resp.read())
            tag = "（⚠ 已停用 SSL 驗證）" if insecure else ""
            print(f"✓ 已下載：{dest.name}（{dest.stat().st_size // 1024} KB）{tag}")
            return True
        except urllib.error.URLError as e:
            msg = str(e)
            if not insecure and ("CERTIFICATE_VERIFY" in msg or "SSL" in msg):
                print(f"  SSL 驗證失敗 → 改試停用驗證重新下載 ({dest.name})")
                continue
            print(f"✗ 下載失敗（{dest.name}）：{type(e).__name__}: {e}")
            return False
        except Exception as e:
            print(f"✗ 下載失敗（{dest.name}）：{type(e).__name__}: {e}")
            return False
    return False


shp_ok = _download(SHP_ZIP_URL, SHP_ZIP)
cdc_ok = _download(CDC_CSV_URL, CDC_CSV)

_DEMO = not (shp_ok and cdc_ok)
if _DEMO:
    print("\n⚠️  部分資料無法下載，後續使用合成示範資料")
    print("    連網後重新執行即可取得真實資料")
else:
    print("\n✅ 資料下載完成")

In [ ]:
# --- Step 2: 讀取縣市邊界 SHP ---
# NLSC SHP 使用 EPSG:3824（TWD97 經緯度），可直接轉換為 WGS84 (EPSG:4326)

# 幾何是否為合成示範。與 _DEMO 分開追蹤：疫情數據下載失敗會設 _DEMO=True，
# 但此時真實地圖幾何可能已經讀進來了；視圖框只該看「幾何」是真是假。
_DEMO_GEOM = False

if not _DEMO:
    # 解壓縮 SHP zip
    if not SHP_DIR.exists():
        with zipfile.ZipFile(SHP_ZIP) as zf:
            zf.extractall(SHP_DIR)
            print(f"解壓縮至 {SHP_DIR}/")

    shp_files = sorted(SHP_DIR.rglob("*.shp"))
    if not shp_files:
        print("⚠️  ZIP 內找不到 .shp，改用示範資料"); _DEMO = True
    else:
        print(f"SHP 檔案：{[f.name for f in shp_files]}")
        gdf = gpd.read_file(shp_files[0])
        # TWD97 → WGS84（方便 matplotlib 直接繪製）
        gdf = gdf.to_crs(epsg=4326)
        print(f"\n縣市數：{len(gdf)}")
        print(f"欄位：{list(gdf.columns)}")

        # 自動偵測縣市名稱欄位（找含有「縣」或「市」的字串欄位）
        county_col = None
        for col in gdf.select_dtypes(include="object").columns:
            vals = gdf[col].dropna().head(10).tolist()
            if any(("縣" in str(v) or "市" in str(v)) for v in vals):
                county_col = col
                break
        if county_col is None:
            county_col = gdf.select_dtypes(include="object").columns[0]
        print(f"\n縣市名稱欄位：{county_col!r}")
        print(gdf[[county_col]].sort_values(county_col).to_string(index=False))

if _DEMO:
    # 無法下載真實邊界時，改用「tile-grid 地圖（tilegram）」示範：
    # 每個縣市 = 一個等大方格、彼此不重疊，位置大致對應台灣地理
    # （北上、南下、離島偏西）。好處：離線也一定畫得出分級著色，
    # 且不會像把方格疊在真實中心點那樣互相蓋住顏色。
    from shapely.geometry import Polygon
    TILE_LAYOUT = {
        "連江縣": (1, 10), "基隆市": (4, 9), "臺北市": (3, 9), "新北市": (3, 8),
        "桃園市": (2, 8),  "宜蘭縣": (4, 8), "新竹市": (2, 7), "新竹縣": (3, 7),
        "金門縣": (0, 6),  "苗栗縣": (2, 6), "臺中市": (3, 6), "花蓮縣": (4, 6),
        "彰化縣": (2, 5),  "南投縣": (3, 5), "澎湖縣": (0, 4), "雲林縣": (2, 4),
        "嘉義市": (2, 3),  "嘉義縣": (3, 3), "臺南市": (2, 2), "高雄市": (2, 1),
        "臺東縣": (3, 1),  "屏東縣": (2, 0),
    }
    DEMO_NAMES = list(TILE_LAYOUT)
    polys = [Polygon([(x, y), (x + 0.92, y), (x + 0.92, y + 0.92), (x, y + 0.92)])
             for x, y in TILE_LAYOUT.values()]
    gdf = gpd.GeoDataFrame({"COUNTYNAME": DEMO_NAMES, "geometry": polys}, crs="EPSG:4326")
    county_col = "COUNTYNAME"
    _DEMO_GEOM = True
    print(f"使用 {len(gdf)} 個縣市方格（tilegram 示範，非真實地理位置）")

In [ ]:
# --- Step 3: 讀取 CDC 退伍軍人病監測資料 ---
# 注意：台/臺 正規化 + 舊縣市名稱（2010 年前）→ 現行名稱

# 台/臺 對照表（官方用字為「臺」；NLSC SHP 也使用「臺」）
# 同時處理 2010 年縣市合併：台北縣→新北市、台中縣/市→臺中市等
TAI_NORMALIZE = {
    # 台 → 臺（4 個縣市有此差異）
    "台北市": "臺北市",  "台中市": "臺中市",
    "台南市": "臺南市",  "台東縣": "臺東縣",
    # 2010 年改制：舊縣市名 → 現行縣市名
    "臺北縣": "新北市",  "台北縣": "新北市",
    "臺中縣": "臺中市",  "台中縣": "臺中市",
    "臺南縣": "臺南市",  "台南縣": "臺南市",
    "高雄縣": "高雄市",  "桃園縣": "桃園市",
}

def normalize_county(name: str) -> str:
    """台/臺 正規化 + 2010 年改制舊名統一。"""
    return TAI_NORMALIZE.get(str(name).strip(), str(name).strip())

if not _DEMO:
    # 讀取 CSV（自動偵測編碼）
    for enc in ("utf-8-sig", "utf-8", "cp950"):
        try:
            raw = pd.read_csv(CDC_CSV, encoding=enc)
            break
        except Exception:
            continue

    raw.columns = [c.strip() for c in raw.columns]
    print("=== CDC 資料欄位 ===")
    print(raw.columns.tolist())
    print(raw.head(3).to_string())

    # 自動偵測欄位名稱（應對不同版本的 CDC OD 格式）
    def _find(candidates):
        for c in candidates:
            if c in raw.columns: return c
        return None

    year_col   = _find(["發病年份","年份","Year","year"])
    county_col_cdc = _find(["縣市","County","county","行政區"])
    cases_col  = _find(["確定病例數","病例數","Cases","cases","個案數"])
    print(f"\n偵測：年份={year_col}, 縣市={county_col_cdc}, 病例={cases_col}")

    if not all([year_col, county_col_cdc, cases_col]):
        print("⚠️  欄位偵測失敗，改用示範資料"); _DEMO = True
    else:
        df = raw[[year_col, county_col_cdc, cases_col]].copy()
        df.columns = ["year", "county", "cases"]
        df["cases"]  = pd.to_numeric(df["cases"], errors="coerce").fillna(0).astype(int)
        df["county"] = df["county"].apply(normalize_county)
        # 只保留有地圖對應的縣市
        valid = set(gdf[county_col].apply(normalize_county))
        df = df[df["county"].isin(valid)].copy()
        print(f"\n共 {len(df)} 筆，年份：{df['year'].min()}–{df['year'].max()}")
        print(f"縣市數：{df['county'].nunique()}")

if _DEMO:
    rng1 = np.random.default_rng(42)
    demo_counties = gdf[county_col].tolist()
    north = {"臺北市","新北市","桃園市","基隆市","新竹市","新竹縣"}
    records = []
    for yr in range(2003, 2025):
        for cn in demo_counties:
            base = 3 if cn in north else 1
            records.append({
                "year": yr, "county": cn,
                "cases": max(0, int(rng1.poisson(base * (1 + (yr-2003)*0.04))))
            })
    df = pd.DataFrame(records)
    print(f"合成示範資料：{len(df)} 筆，{df['year'].min()}–{df['year'].max()}")

In [ ]:
# --- Step 4: ID 比對除錯（縣市名稱一致性檢查）---
# 這是製作 Choropleth 前最重要的步驟：確認 SHP 和 CSV 的縣市名稱完全相符

shp_counties  = set(gdf[county_col].apply(normalize_county))
data_counties = set(df["county"].unique())

print("=== 縣市 ID 比對結果 ===")
only_in_shp  = sorted(shp_counties - data_counties)
only_in_data = sorted(data_counties - shp_counties)
matched      = sorted(shp_counties & data_counties)

if only_in_shp:
    print(f"\n⚠️  只在 SHP（地圖將呈現空白）：{only_in_shp}")
else:
    print("\n✅ SHP 所有縣市在 CDC 資料中都有對應")

if only_in_data:
    print(f"\n⚠️  只在 CDC 資料（不會顯示在地圖）：{only_in_data}")
    print("    （可能為「不詳」、「外縣市」等非縣市代碼，已在前一步過濾）")
else:
    print("✅ CDC 資料所有縣市在 SHP 中都有對應")

print(f"\n✅ 成功匹配 {len(matched)} 個縣市")

In [ ]:
# --- Step 5: 按年度 × 縣市彙總，計算每十萬人發生率 ---
# 退伍軍人病通報率低，絕對病例數小；用發生率（per 100k）比較各縣市更公平

# 近似縣市人口（2023 年估計，單位：人）
COUNTY_POP = {
    "臺北市": 2_530_000, "新北市": 4_036_000, "桃園市": 2_307_000,
    "臺中市": 2_854_000, "臺南市": 1_876_000, "高雄市": 2_761_000,
    "基隆市":   370_000, "新竹市":   454_000, "新竹縣":   574_000,
    "苗栗縣":   543_000, "彰化縣": 1_284_000, "南投縣":   487_000,
    "雲林縣":   673_000, "嘉義市":   268_000, "嘉義縣":   504_000,
    "屏東縣":   822_000, "宜蘭縣":   461_000, "花蓮縣":   325_000,
    "臺東縣":   223_000, "澎湖縣":   107_000, "金門縣":   143_000,
    "連江縣":    14_000,
}

annual = df.groupby(["year","county"])["cases"].sum().reset_index()
annual["population"]   = annual["county"].map(COUNTY_POP).fillna(500_000)
annual["rate_per_100k"] = (annual["cases"] / annual["population"] * 100_000).round(3)

print(f"年份範圍：{annual['year'].min()}–{annual['year'].max()}")
print(f"總通報病例：{annual['cases'].sum()} 例")
print("\n累計發生率最高前 5 縣市（每十萬人）：")
top5 = (annual.groupby("county")
              .agg(total_cases=("cases","sum"), avg_rate=("rate_per_100k","mean"))
              .nlargest(5,"avg_rate"))
print(top5.round(3).to_string())

In [ ]:
# --- Step 6: 靜態 Choropleth（最新年度）---
# 視圖範圍：真實幾何 → 台灣本島固定框（剪掉東沙、太平島等遠距離島）；
#          示範幾何（tilegram）→ 依資料範圍自動框選，保證一定畫得出來。
TW_CLIP = (119.323286, 122.128611, 21.739091, 25.621716)  # (lon_min, lon_max, lat_min, lat_max)

def view_window(geo, fallback=TW_CLIP, pad=0.04):
    """由幾何範圍推導安全視圖框；範圍含 NaN 或退化（空、線）時退回 fallback。"""
    valid = geo.geometry.dropna()
    if len(valid) == 0:
        return fallback
    minx, miny, maxx, maxy = valid.total_bounds
    if not np.all(np.isfinite([minx, miny, maxx, maxy])) or minx >= maxx or miny >= maxy:
        return fallback
    mx, my = (maxx - minx) * pad, (maxy - miny) * pad
    return (minx - mx, maxx + mx, miny - my, maxy + my)

latest_year = int(annual["year"].max())
latest = annual[annual["year"] == latest_year].copy()

# 合併地圖與統計資料
gdf_plot = gdf.copy()
gdf_plot["county_norm"] = gdf_plot[county_col].apply(normalize_county)
gdf_merged = gdf_plot.merge(
    latest[["county", "rate_per_100k", "cases"]],
    left_on="county_norm", right_on="county", how="left"
)
gdf_merged["rate_per_100k"] = gdf_merged["rate_per_100k"].fillna(0)

# 依「幾何來源」決定視圖框（不是依 _DEMO；真實幾何即使配示範疫情也要用固定框）
LON_MIN, LON_MAX, LAT_MIN, LAT_MAX = view_window(gdf_merged) if _DEMO_GEOM else TW_CLIP

fig, ax = plt.subplots(figsize=(7, 9), dpi=150)
fig.patch.set_facecolor("#FAF8F3")
ax.set_facecolor("#FAF8F3")

gdf_merged.plot(
    column="rate_per_100k",
    ax=ax,
    cmap="Reds",
    legend=True,
    legend_kwds={
        "label": "發生率（每十萬人）",
        "orientation": "horizontal",
        "shrink": 0.6,
        "pad": 0.01,
    },
    edgecolor="white",
    linewidth=0.5,
    missing_kwds={"color": "#E8E5DF", "label": "無資料"},
)
# tilegram 示範模式：在每格標上縣市名（真實地圖則不加，避免壓字）
if _DEMO_GEOM:
    import matplotlib.patheffects as _pe
    _halo = [_pe.withStroke(linewidth=1.8, foreground="white")]
    for _, _r in gdf_merged.iterrows():
        _c = _r.geometry.centroid
        ax.annotate(_r[county_col], (_c.x, _c.y), ha="center", va="center",
                    fontsize=6, color="#1A1A1A", path_effects=_halo)
# 限縮視圖（ax.clear 不影響此處；靜態圖只畫一次）
ax.set_xlim(LON_MIN, LON_MAX)
ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_aspect("equal")
_src_note = (
    "疾管署開放資料" if not _DEMO
    else "示範資料，tilegram 示意地理" if _DEMO_GEOM
    else "示範資料，真實縣市界"
)
ax.set_title(
    f"{latest_year} 年台灣退伍軍人病發生率（每十萬人）\n"
    f"（資料來源：{_src_note}）",
    fontsize=12, pad=10
)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"\n→ {latest_year} 年共通報 {latest['cases'].sum()} 例")
print("→ 前 3 名縣市：")
for _, row in latest.nlargest(3, "rate_per_100k").iterrows():
    print(f"   {row['county']}：{row['cases']} 例（{row['rate_per_100k']:.3f}/10萬）")

In [ ]:
# --- Step 7: 年度動畫 Choropleth（FuncAnimation → GIF）---
# 展示 2003 年以來各縣市發生率的時間變化
# 視圖範圍沿用 Step 6（LON_MIN..LON_MAX, LAT_MIN..LAT_MAX，已依幾何來源決定）

years = sorted(annual["year"].unique())
# 色軸上限用 95 百分位（避免少數高值壓縮整體色彩）
vmax  = annual["rate_per_100k"].quantile(0.95)
vmax  = max(vmax, 0.1)  # 避免 vmax = 0

gdf_base = gdf.copy()
gdf_base["county_norm"] = gdf_base[county_col].apply(normalize_county)

fig, ax = plt.subplots(figsize=(5, 7), dpi=100)
fig.patch.set_facecolor("#FAF8F3")

def _update(year):
    ax.clear()
    ax.set_facecolor("#FAF8F3")
    yr_data = annual[annual["year"] == year][["county","rate_per_100k"]]
    merged  = gdf_base.merge(yr_data, left_on="county_norm", right_on="county", how="left")
    merged["rate_per_100k"] = merged["rate_per_100k"].fillna(0)
    merged.plot(
        column="rate_per_100k", ax=ax,
        cmap="Reds", vmin=0, vmax=vmax,
        edgecolor="white", linewidth=0.5,
        legend=False,
    )
    # 每幀都要重新設定 xlim/ylim（ax.clear() 會清掉）
    ax.set_xlim(LON_MIN, LON_MAX)
    ax.set_ylim(LAT_MIN, LAT_MAX)
    ax.set_aspect("equal")
    ax.set_title(f"台灣退伍軍人病發生率\n{year} 年（每十萬人）", fontsize=11)
    ax.axis("off")

anim = FuncAnimation(fig, _update, frames=years, interval=800, repeat=True)

# 存成 GIF（使用 pillow writer，dpi=100 適合網頁內嵌）
gif_path = DATA_DIR / "tw_legionella_anim.gif"
try:
    anim.save(str(gif_path), writer="pillow", fps=1, dpi=100)
    plt.close()
    print(f"✓ 動畫已存至：{gif_path}")
    print(f"  共 {len(years)} 幀，涵蓋 {years[0]}–{years[-1]} 年")
    print(f"  解析度：100 dpi（{gif_path.stat().st_size // 1024} KB）")
    display(IPyImage(filename=str(gif_path)))
except Exception as e:
    plt.close()
    print(f"⚠️  無法儲存 GIF：{e}")

## Step 8：換上真實台灣地圖（內建 GeoJSON）

前面的 tilegram 是離線示意用的方格圖。這裡改用專案**內建的真實台灣縣市界線** `data/geojson/county_smooth_inset.geojson`：邊界已平滑化，離島（馬祖、金門、澎湖）以 **inset** 方式縮放排版在本島西側，不必連網就能畫出真正的台灣地圖。

- **內建於 repo**：離線、Colab、書本建置都能用，一定畫得出來（這正是我們一開始想要、卻無法即時從政府網站下載的東西）。
- **疫情資料沿用** Step 5 的 `annual`（真實或示範皆可），以正規化後的 `COUNTYNAME` 做 JOIN。

In [ ]:
# --- Step 8: 真實台灣縣市 Choropleth（內建 GeoJSON，離線可用）---
# tilegram 是離線示意；這裡改用 repo 內建的真實縣市界線（已平滑、離島 inset 排版）。
import matplotlib.patheffects as pe

def _locate(rel):
    """從 cwd 逐層往上找檔案，兼容 book 建置 / standalone / Colab 的工作目錄。"""
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (base / rel).exists():
            return base / rel
    return pathlib.Path(rel) if pathlib.Path(rel).exists() else None

geojson_path = _locate("data/geojson/county_smooth_inset.geojson")
if geojson_path is None:
    print("⚠️  找不到內建 GeoJSON（data/geojson/county_smooth_inset.geojson），略過真實地圖")
else:
    real = gpd.read_file(geojson_path)
    real["county_norm"] = real["COUNTYNAME"].apply(normalize_county)

    # 沿用 Step 5 算好的 annual，取最新年度發生率（真實或示範疫情皆可）
    _yr = int(annual["year"].max())
    _rate = annual[annual["year"] == _yr][["county", "rate_per_100k", "cases"]]
    real = real.merge(_rate, left_on="county_norm", right_on="county", how="left")
    real["rate_per_100k"] = real["rate_per_100k"].fillna(0)

    fig, ax = plt.subplots(figsize=(7, 8), dpi=150)
    fig.patch.set_facecolor("#FAF8F3")
    ax.set_facecolor("#FAF8F3")
    real.plot(
        column="rate_per_100k", ax=ax, cmap="Reds",
        edgecolor="#8A8A8A", linewidth=0.4, legend=True,
        legend_kwds={"label": "發生率（每十萬人）", "orientation": "horizontal",
                     "shrink": 0.6, "pad": 0.01},
        missing_kwds={"color": "#E8E5DF", "label": "無資料"},
    )
    # 小面積縣市（臺北市、基隆市、新竹市、嘉義市）的標籤比縣市本身還大，
    # 直接標會壓到隔壁縣市的色層；改用「引線標註」把標籤拉到鄰近空白海域，
    # 再用細線指回該縣市（其餘縣市維持原地標註）。
    _CALLOUT = {
        "新竹市": (119.72, 24.90), "嘉義市": (119.72, 23.60),  # 拉到台灣海峽空隙
        "臺北市": (121.30, 25.52), "基隆市": (122.18, 25.35),  # 拉到北部／東北海域
    }
    _halo = [pe.withStroke(linewidth=1.8, foreground="white")]
    for _, _row in real.iterrows():
        _il = _row.get("inset_label")
        _label = _il if isinstance(_il, str) and _il else _row["COUNTYNAME"]
        _pt = _row.geometry.representative_point()
        _anchor = _CALLOUT.get(_row["COUNTYNAME"])
        if _anchor is None:
            ax.annotate(_label, (_pt.x, _pt.y), ha="center", va="center",
                        fontsize=5.5, color="#1A1A1A", path_effects=_halo)
        else:
            ax.annotate(_label, xy=(_pt.x, _pt.y), xytext=_anchor, textcoords="data",
                        ha="center", va="center", fontsize=5.5, color="#1A1A1A",
                        path_effects=_halo,
                        arrowprops=dict(arrowstyle="-", lw=0.5, color="#6B6B6B",
                                        shrinkA=0, shrinkB=2))
    # inset 排版已把離島移到本島西側；右／上多留白給引線標籤
    minx, miny, maxx, maxy = real.total_bounds
    ax.set_xlim(minx - 0.1, maxx + 0.35)
    ax.set_ylim(miny - 0.1, maxy + 0.18)
    ax.set_aspect("equal")
    ax.set_title(
        f"{_yr} 年台灣退伍軍人病發生率（每十萬人）\n"
        f"（真實縣市地圖・離島以 inset 呈現）",
        fontsize=12, pad=10,
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"→ 使用內建真實地圖：{geojson_path}")
    print(f"→ {real['COUNTYNAME'].nunique()} 個縣市；{_yr} 年發生率最高：")
    for _, _r in real.nlargest(3, "rate_per_100k").iterrows():
        print(f"   {_r['COUNTYNAME']}：{_r['rate_per_100k']:.3f}/10萬")

## 小結

| 步驟 | 學到的技能 |
|---|---|
| 下載政府 SHP + CSV | `urllib.request` 帶 User-Agent 繞過 403 |
| 讀取 SHP | `geopandas.read_file()` + `to_crs(epsg=4326)` |
| 自動偵測欄位 | 字串搜尋法尋找縣市名稱欄位 |
| 台/臺 正規化 | `str.replace` + 改制前後名稱對照表 |
| ID 比對除錯 | `set.difference()` 找出不匹配的縣市 |
| 靜態 Choropleth | `gdf.merge()` + `gdf.plot(column=..., cmap=...)` |
| 動態 Choropleth | `FuncAnimation` → `anim.save(..., writer="pillow")` |
| 真實縣市地圖 | 內建 GeoJSON（縣市界＋離島 inset）＋ `gpd.merge` |

### 關鍵觀念

- **台/臺**：政府公文、CDC、NLSC 官方用「臺」；網路文章常見「台」。JOIN 前必須正規化。
- **2010 年改制**：台北縣 → 新北市；台中縣/市 → 臺中市；2003 年的資料用舊名稱，需要對照表轉換。
- **每十萬人發生率 vs 絕對病例數**：人口多的縣市病例多不代表風險高，一定要用人口標準化。
- **合成示範資料**：若網路不通，notebook 自動切換合成資料，概念相同，數字不代表真實情況。
- **內建真實地圖**：把縣市 GeoJSON 放進 repo，離線也能畫出真正的台灣地圖，不必依賴政府網站即時下載。